In [1]:
import math
import re
import string
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from matplotlib import pyplot as plt
from collections import Counter

random.seed(42)



import random
import nltk
from nltk.corpus import gutenberg

In [2]:
num_sentences = 10      #None

---

# templated

In [3]:
def print_sentence(tokens):

    if not tokens:
        print("")
        return ""

    # Drop boundary markers if present
    start = 1 if tokens and tokens[0] == "<s>" else 0
    end = -1 if tokens and tokens[-1] == "</s>" else len(tokens)
    core = tokens[start:end]

    if not core:
        print("")
        return ""

    # Capitalize first token
    out = core[0].capitalize()

    # Process remaining tokens with spacing rules
    for i, t in enumerate(core[1:], start=1):
        is_last = (i == len(core) - 1)

        # Last-token punctuation rule
        if is_last and t in {".", "?", "!"}:
            out += t
            continue

        # General punctuation attaches with no space (e.g., ',', ':', ';', ')', etc.)
        if t in string.punctuation:
            out += t
            continue

        # Apostrophe-start tokens like "'s", "'d", "'ll"
        if t.startswith("'"):
            out += t
            continue

        # Capitalize special standalone words
        if t.lower() in {"i", "god"}:
            out += " " + t.capitalize()
            continue

        # Default: add a space before normal words
        out += " " + t

    # print(out)
    return out


----
# prep & data

In [4]:
nltk.download("gutenberg", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

True

In [5]:
austen_files = [
    "austen-emma.txt",
    "austen-persuasion.txt",
    "austen-sense.txt",
]


# --------------------------------------------------
# Combine sentences from all three novels
# --------------------------------------------------

all_sentences = []

random_seed = 42

for filename in austen_files:
    all_sentences.extend(
        gutenberg.sents(filename)
    )

rng = random.Random(random_seed)
rng.shuffle(all_sentences)

if num_sentences is None:
    selected_sentences = all_sentences
else:
    selected_sentences = all_sentences[:num_sentences]

    
# --------------------------------------------------
# Normalize and add sentence-boundary markers
# --------------------------------------------------

tokenized_sentences = []

for sentence in selected_sentences:

    tokens = [
        token.lower()
        for token in sentence
        if token.strip()
    ]

    if tokens:
        tokenized_sentences.append(
            ["<s>"] + tokens + ["</s>"]
        )

### test

In [6]:
# for k in range(10):
#     if num_sentences != None and (k >= num_sentences):
#         break
#     print_sentence(tokenized_sentences[k]) 
#     print()
#     print(tokenized_sentences[k]) 
#     print()

In [7]:
# Basic corpus statistics
# --------------------------------------------------

# Remove sentence-boundary markers
all_tokens = [
    token
    for sentence in tokenized_sentences
    for token in sentence
    if token not in {"<s>", "</s>"}
]


def is_word(token):
    return (
        token.isalpha()
        or token.replace("'", "").isalpha()
    )


words = [
    token
    for token in all_tokens
    if is_word(token)
]


num_tokens = len(all_tokens)
num_unique_tokens = len(set(all_tokens))

num_words = len(words)
num_unique_words = len(set(words))

sentence_lengths = [
    len(sentence) - 2       # exclude <s> and </s>
    for sentence in tokenized_sentences
]


print(f"Total number of sentences: {len(tokenized_sentences):,}")
print(f"Total number of tokens:    {num_tokens:,}")
print(f"Number of unique tokens:   {num_unique_tokens:,}")
print(f"Total number of words:     {num_words:,}")
print(f"Number of unique words:    {num_unique_words:,}")

Total number of sentences: 10
Total number of tokens:    362
Number of unique tokens:   196
Total number of words:     305
Number of unique words:    183


#### zipf's law

In [8]:
# # Word frequencies
# # --------------------------------------------------

# # Count words only, excluding punctuation and
# # sentence-boundary markers
# word_counts = Counter(
#     token
#     for sentence in tokenized_sentences
#     for token in sentence
#     if is_word(token)
# )

# # Frequencies sorted by rank
# freqs = np.array([
#     freq
#     for token, freq in word_counts.most_common()
# ])

# ranks = np.arange(1, len(freqs) + 1)

# plt.figure(figsize=(8, 5))
# plt.plot(ranks, freqs, label="Corpus data")

# # Reference line: slope = -1 (Zipf's law)
# # Match scale by anchoring at the first frequency
# ref_line = freqs[0] / ranks
# plt.plot(
#     ranks,
#     ref_line,
#     "--",
#     label="Reference slope = -1"
# )

# plt.xscale("log")
# plt.yscale("log")
# plt.title("Word Frequencies vs. Rank (Zipf's Law)")
# plt.xlabel("Rank of word")
# plt.ylabel("Frequency")
# plt.grid(True, which="both", ls="--", alpha=0.6)
# plt.legend()
# plt.tight_layout()
# plt.show()

In [9]:
# # Sentence lengths
# # --------------------------------------------------

# # Subtract the <s> and </s> boundary tokens
# sentence_lengths = [
#     len(sentence) - 2
#     for sentence in tokenized_sentences
# ]

# plt.figure(figsize=(8, 4))
# plt.hist(
#     sentence_lengths,
#     bins=range(1, max(sentence_lengths) + 2),
#     color="skyblue",
#     edgecolor="black"
# )

# plt.title("Histogram of Sentence Lengths")
# plt.xlabel("Sentence length (in tokens)")
# plt.ylabel("Number of sentences")
# plt.tight_layout()
# plt.show()

# print(
#     f"Average sentence length: "
#     f"{np.mean(sentence_lengths):.2f} tokens"
# )

# print(
#     f"Max sentence length:     "
#     f"{max(sentence_lengths)} tokens"
# )

---

# p1

### **todo**

1. Build N-grams
1. Create dictionary 
    1. N-1 for left
    1. final toke as right
    1. make `next_word_lists`

1. next token frequency into log probabilty
1. make dictionary `get_next_word`
1. sanity check
1. generate 5 possible next token and convert back from log.



**graded**


1. most likely next token on 3-gram of `<s>`,`the`.

1. answer question 




In [10]:
def get_ngrams(sent, N):
    return [ tuple(sent[k:k+N]) for k in range(len(sent) - N + 1) ]



In [11]:
all_bigrams  = []

for sentence in tokenized_sentences:
    sentence_bigrams  = get_ngrams(sentence, 2) 
    all_bigrams.extend(sentence_bigrams )


all_trigrams = []

for sentence in tokenized_sentences:
    sentece_trigrams = get_ngrams(sentence, 3)
    all_trigrams.extend(sentece_trigrams)

# print(all_bigrams [:10])

In [12]:
# bigram_counts = (
#     pd.DataFrame(all_bigrams, columns=["previous", "word"])
#     .groupby(["previous", "word"])
#     .size()
#     .reset_index(name="count")
#     .sort_values("count", ascending=False)
# )

# print(bigram_counts.head(10))

In [13]:
def get_next_word_dict(ngrams):
    next_word_dict = {}

    for ng in ngrams:
        left_context = ng[:-1]
        next_word = ng[-1]

        if left_context not in next_word_dict:
            next_word_dict[left_context] = []

        next_word_dict[left_context].append(next_word)

    return next_word_dict


next_word_list = get_next_word_dict(all_bigrams)
next_word_list.update(get_next_word_dict(all_trigrams))

##_3. Inspect each context type
print(next_word_list.get(("she",), []))        #__Bigram context
print(next_word_list.get(("she", "was"), []))  #__Trigram context

['went', 'had', 'heard', 'heard', 'was', 'were']
['met']


In [14]:
##_1. Convert next-token counts to probabilities
def tokens2probs(ws):
    k = len(ws)
    return {
        word: count / k
        for word, count in Counter(ws).items()
    }

##_2. Build probability distributions for every context
def get_PLM(D):
    return {
        context: tokens2probs(next_words)
        for context, next_words in D.items()
    }

##_3. Calculate probabilities separately for each context
get_next_word = get_PLM(next_word_list)


display(get_next_word[('the',)])
display(get_next_word[('<s>',)])
display(get_next_word[('<s>','he')])

{'door': 0.07692307692307693,
 'house': 0.07692307692307693,
 'morning': 0.07692307692307693,
 'worst': 0.07692307692307693,
 'fear': 0.07692307692307693,
 'passage': 0.07692307692307693,
 'instant': 0.07692307692307693,
 'stairs': 0.07692307692307693,
 'kellynch': 0.07692307692307693,
 'honour': 0.07692307692307693,
 'family': 0.07692307692307693,
 'weather': 0.07692307692307693,
 'intelligence': 0.07692307692307693}

{"'": 0.1,
 '"`': 0.1,
 'a': 0.1,
 '"': 0.1,
 'he': 0.2,
 'she': 0.1,
 '--': 0.1,
 'his': 0.1,
 'if': 0.1}

{'did': 0.5, 'lived': 0.5}

In [15]:
#1. Create a reproducible random generator
rng = np.random.default_rng(42)

##_2. Define the three fixed contexts
contexts = [
    ("<s>",),
    ("the",),
    ("<s>", "he"),
    (("<s>", "the"),)
]

##_3. Sample five next tokens for each context
for context in contexts:
    distribution = get_next_word.get(context, {})

    if not distribution:
        print(context, "→ Context not found")
        continue

    #__Keep tokens and their log probabilities in matching order
    tokens = list(distribution.keys())
    log_probs = np.array(list(distribution.values()))

    #__Convert natural-log probabilities back to probabilities
    probs = np.exp(log_probs)
    probs = probs / probs.sum()

    #__Draw five times; repeated tokens are allowed
    samples = rng.choice(
        tokens,
        size=5,
        replace=True,
        p=probs
    )

    print(context, "→", samples.tolist())

('<s>',) → ['--', '"', 'his', '--', "'"]
('the',) → ['intelligence', 'honour', 'family', 'house', 'passage']
('<s>', 'he') → ['did', 'lived', 'lived', 'lived', 'did']
(('<s>', 'the'),) → Context not found


In [16]:

##_1. Create a reproducible random generator
rng = np.random.default_rng(42)

##_2. Define the three fixed contexts
contexts = [
    ("<s>",),
    ("the",),
    ("<s>", "he"),
]

##_3. Sample five next tokens for each context
for context in contexts:
    distribution = get_next_word.get(context, {})

    if not distribution:
        print(context, "→ Context not found")
        continue

    #__Keep tokens and their log probabilities in matching order
    tokens = list(distribution.keys())
    log_probs = np.array(list(distribution.values()))

    #__Convert natural-log probabilities back to probabilities
    probs = np.exp(log_probs)
    probs = probs / probs.sum()

    #__Draw five times; repeated tokens are allowed
    samples = rng.choice(
        tokens,
        size=5,
        replace=True,
        p=probs
    )

    print(context, "→", samples.tolist())

('<s>',) → ['--', '"', 'his', '--', "'"]
('the',) → ['intelligence', 'honour', 'family', 'house', 'passage']
('<s>', 'he') → ['did', 'lived', 'lived', 'lived', 'did']


In [17]:
##_1. Choose a context and retrieve its distribution
p1_context = ("<s>", "he")

p1_distribution = get_next_word.get(p1_context, {})

##_2. Select the most likely next token
if not p1_distribution:
    print(p1_context, "→ Context not found")
else:
    next_token = max(p1_distribution, key=p1_distribution.get)

    print("Most likely next token:", next_token)
    print("Log probability:", p1_distribution[next_token])

Most likely next token: did
Log probability: 0.5


----

# p2




In [18]:
def sample_next_token(PLM, left_context, temperature=1.0):

    #__1. Start with a trigram context: last two tokens
    context = tuple(left_context[-2:])

    #__2. Back off by dropping the oldest token
    distribution = PLM.get(context, {})

    while not distribution and context:
        context = context[1:]
        distribution = PLM.get(context, {})

    if not distribution:
        raise ValueError("No distribution found for this context")

    #__3. Retrieve tokens and matching log probabilities
    tokens = list(distribution.keys())
    log_probs = np.array(list(distribution.values()))

    #__4. Apply temperature in log space
    scaled_logs = log_probs / temperature

    #__5. Convert to ordinary probabilities; normalize
    weights = np.exp(scaled_logs - scaled_logs.max())
    probs = weights / weights.sum()

    #__6. Sample and return one token
    return str(rng.choice(tokens, p=probs))

In [19]:
def generate_sentence(PLM, temperature=1.0, max_tokens=40):
    tokens = ["<s>"]

    for _ in range(max_tokens):
        next_token = sample_next_token(
            PLM,
            left_context=tokens,
            temperature=temperature
        )

        tokens.append(next_token)

        if next_token == "</s>":
            break

    return tokens

    # return " ".join(
    #     token for token in tokens
    #     if token not in ("<s>", "</s>")
    #     )


#__Start with a bigram; use trigrams once two tokens are available
print(generate_sentence(get_next_word, temperature=1.1))

['<s>', 'a', 'poet', 'in', 'love', 'must', 'be', 'encouraged', 'in', 'both', 'capacities', ',', 'or', 'neither', '.', '</s>']


In [20]:
p2_test1 = generate_sentence(get_next_word, temperature=0.8)
display(p2_test1)
print_sentence(p2_test1)

['<s>',
 '--',
 'and',
 'when',
 'mrs',
 '.',
 'bates',
 'was',
 'saying',
 'something',
 'to',
 'emma',
 ',',
 'whispered',
 'farther',
 ',',
 '"',
 'beg',
 'her',
 'to',
 'walk',
 'up',
 ';"--',
 'and',
 'a',
 'moment',
 'afterwards',
 'she',
 'was',
 'met',
 'on',
 'the',
 'stairs',
 'by',
 'jane',
 'herself',
 ',',
 'coming',
 'eagerly',
 'forward',
 ',']

'-- and when mrs. bates was saying something to emma, whispered farther," beg her to walk up ;"-- and a moment afterwards she was met on the stairs by jane herself, coming eagerly forward,'

In [21]:
temp_list = [.2 , 1 , 5]

p2list = {}

for t in temp_list:
    sampler = {}
    for i in range(10):
        test_sent = generate_sentence(get_next_word, temperature=t)
        sampler[i] = print_sentence(test_sent)
    p2list[t] = (sampler)




    


In [22]:
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_columns", None,
):
    display(pd.DataFrame(p2list).melt())

,variable,value
0,0.2,"A poet in love must be encouraged in both capacities, or neither."
1,0.2,"' my dear madam ,' I always say to her,' you must make yourself easy."
2,0.2,"-- and when mrs. goddard' s, and send up her name .-- she heard nothing but the instant reply of,"" we do not say a word of a certain young physician from windsor .-- oh"
3,0.2,He did consent.
4,0.2,"' my dear madam ,' I always say to her,' you must make yourself easy."
5,0.2,""" thank you, willoughby."
6,0.2,"' my dear madam ,' I always say to her,' you must make yourself easy."
7,0.2,"""` mr."
8,0.2,"""` mr."
9,0.2,He did consent.


---
# p3

In [23]:

def sample_next_token_v2(PLM, left_context, temperature=1.0):
    #__1. Trigram context first, then back off to bigram
    context = tuple(left_context[-2:])
    distribution = PLM.get(context, {})

    while not distribution and context:
        context = context[1:]
        distribution = PLM.get(context, {})

    if not distribution:
        raise ValueError("No distribution found for this context")

    #__2. Tokens and original (untempered) log probabilities
    tokens = list(distribution.keys())
    log_probs = np.array(list(distribution.values()))

    #__3. Temperature only affects sampling weights
    scaled_logs = log_probs / temperature
    weights = np.exp(scaled_logs - scaled_logs.max())
    probs = weights / weights.sum()

    #__4. Sample one token
    next_word = str(rng.choice(tokens, p=probs))

    #__5. Return original log probability, not tempered
    next_word_log_prob = distribution[next_word]

    return next_word, next_word_log_prob

def generate_sentence_v2(PLM, temperature=1.0, max_tokens=40):
    tokens = ["<s>"]
    sum_log_probs = 0.0

    for _ in range(max_tokens):
        next_word, next_word_log_prob = sample_next_token_v2(
            PLM, tokens, temperature
        )

        tokens.append(next_word)
        sum_log_probs += next_word_log_prob

        if next_word == "</s>":
            break

    #__Average log prob per generated token (excludes initial <s>)
    ALP = sum_log_probs / (len(tokens) - 1)
    PP = np.exp(-ALP)

    return tokens, PP



In [24]:

tokens, PP = generate_sentence_v2(get_next_word, temperature=1.0)
display(tokens, PP)

['<s>',
 'she',
 'went',
 '--',
 'she',
 'had',
 'driven',
 'once',
 'unsuccessfully',
 'to',
 'the',
 'door',
 ',',
 'but',
 'all',
 'the',
 'worst',
 'of',
 'her',
 'sufferings',
 'had',
 'been',
 'unsuspected',
 '.--',
 'the',
 'fear',
 'of',
 'being',
 'still',
 'unwelcome',
 ',',
 'determined',
 'her',
 ',',
 'though',
 'assured',
 'of',
 'their',
 'being',
 'at',
 'home']

np.float64(0.4055545050633205)

In [25]:

##_1. Settings
temp_list = [0.2, 1, 5]
n_sentences = 10

##_2. Collect one row per generated sentence
rows = []

for t in temp_list:
    for i in range(n_sentences):
        sentence_tokens, sentence_pp = generate_sentence_v2(
            get_next_word,
            temperature=t
        )

        rows.append({
            'temp': t,
            'sentence': print_sentence(sentence_tokens),
            'pp': sentence_pp
        })

##_3. Build the DataFrame once
p3_compare = pd.DataFrame(rows)


with pd.option_context(
    "display.max_colwidth", None,
    "display.max_columns", None,
):
    display(p3_compare)
    


,temp,sentence,pp
0,0.2,""" thank you, willoughby.",0.418353
1,0.2,"If she were to meet miss bates in her way !-- and upon its beginning to rain, emma was obliged to expect that the weather would be detaining her at mrs. bates was saying something to emma,",0.390628
2,0.2,""" thank you, willoughby.",0.418353
3,0.2,"He lived about a mile from highbury, was a frequent visitor, and always welcome, and at this time more welcome than usual, as coming directly from their mutual connexions in london.",0.402198
4,0.2,He did consent.,0.477114
5,0.2,"If she were to meet miss bates in her way !-- and upon its beginning to rain, emma was obliged to expect that the weather would be detaining her at mrs. bates was saying something to emma,",0.390628
6,0.2,"He lived about a mile from highbury, was a frequent visitor, and send up her name .-- she heard patty announcing it; but no such bustle succeeded as poor miss bates had before made so happily intelligible",0.397027
7,0.2,"' my dear madam ,' I always say to her,' you must make yourself easy.",0.396010
8,0.2,"If she were to meet miss bates had before made so happily intelligible .-- no; she heard nothing but the instant reply of,"" beg her to walk up ;""-- and a moment afterwards she was met on",0.390628
9,0.2,"His chance for the kellynch estate was something, but all the worst of her were felt sufficient .-- emma had never seen her look so well, so engaging.",0.402776


In [26]:
pp_summary = (
    p3_compare
    .groupby('temp')
    .agg(
        mean_pp=('pp', 'mean')
        , median_pp=('pp', 'median')
        , min_pp=('pp', 'min')
        , max_pp=('pp', 'max')
    )
    .reset_index()
)

display(pp_summary)

,temp,mean_pp,median_pp,min_pp,max_pp
0,0.2,0.408372,0.399613,0.390628,0.477114
1,1.0,0.431092,0.434739,0.390628,0.477114
2,5.0,0.430837,0.433129,0.390628,0.477114


---
# p4

In [28]:
def extend_one_token(PLM, state = (["<s>"], 0.0), temperature=1.0):
    #__state = (tokens_so_far, sum_log_probs)
    tokens_so_far, sum_log_probs = state

    #__Trigram context first; back off to bigram if missing
    next_word, next_word_log_prob = sample_next_token_v2(
        PLM,
        left_context=tokens_so_far,
        temperature=temperature
    )

    #__Build new state; do not modify the input list
    new_tokens = tokens_so_far + [next_word]

    #__Add the original, untempered log prob
    new_sum_log_probs = sum_log_probs + next_word_log_prob

    return new_tokens, new_sum_log_probs


##_Quick test
state = (["<s>"], 0.0)
new_state = extend_one_token(get_next_word, state, temperature=1.0)
print(new_state)

(['<s>', '"'], 0.1)


In [29]:
def beam_search(PLM, B=5, C=5, temperature=1.0, max_tokens=40, n_finished=10):
    PQ = [(["<s>"], 0.0)]
    Finished = []

    for step in range(max_tokens):
        candidates = []

        #__1. C one-token continuations for every partial sentence
        for state in PQ:
            for _ in range(C):
                new_tokens, new_slp = extend_one_token(PLM, state, temperature)

                #__2. Finished: compute perplexity, move to Finished
                if new_tokens[-1] == "</s>":
                    ALP = new_slp / (len(new_tokens) - 1)
                    PP = np.exp(-ALP)
                    Finished.append((new_tokens, PP))
                #__3. Otherwise keep as unfinished
                else:
                    candidates.append((new_tokens, new_slp))

        #__Optional: remove exact duplicate token lists
        candidates = list({tuple(t): (t, s) for t, s in candidates}.values())

        #__4-5. Rank by log prob; keep best B
        PQ = sorted(candidates, key=lambda x: x[1], reverse=True)[:B]

        #__Stop rules
        if len(Finished) >= n_finished or not PQ:
            break

    return Finished[:n_finished], PQ



In [30]:

finished, leftover = beam_search(get_next_word, B=5, C=5, temperature=1.0)

for tokens, PP in finished:
    print(round(PP, 2), " ".join(tokens[1:-1]))

0.46 "` mr .
0.46 "` mr .
0.46 "` mr .
0.46 "` mr .
0.46 "` mr .
0.42 " thank you , willoughby .
0.42 " thank you , willoughby .
0.42 " thank you , willoughby .
0.42 " thank you , willoughby .
0.42 " thank you , willoughby .


In [31]:

##_1. Temperatures to compare
temp_list = [0.2, 1.0, 5.0]

##_2. Run beam search once per temperature
rows = []

for t in temp_list:
    finished, leftover = beam_search(
        get_next_word,
        B=5,
        C=5,
        temperature=t,
        max_tokens=40,
        n_finished=10
    )

    for rank, (tokens, PP) in enumerate(finished, start=1):
        rows.append({
            "temperature": t,
            "rank": rank,
            "sentence": " ".join(tokens[1:-1]),
            "n_tokens": len(tokens) - 1,
            "perplexity": round(PP, 2),
        })

##_3. Combine results
df_beam = pd.DataFrame(rows)

##_4. Show all sentences, one row each
with pd.option_context(
    "display.max_colwidth", 100,
    "display.max_columns", None,
):
    display(df_beam)

##_5. Compare temperatures: count and average perplexity
display(
    df_beam
    .groupby("temperature")
    .agg(
        n_sentences=("sentence", "size"),
        avg_perplexity=("perplexity", "mean"),
        avg_tokens=("n_tokens", "mean"),
    )
    .round(2)
)

,temperature,rank,sentence,n_tokens,perplexity
0,0.2,1,"""` mr .",4,0.46
1,0.2,2,"""` mr .",4,0.46
2,0.2,3,"""` mr .",4,0.46
3,0.2,4,"""` mr .",4,0.46
4,0.2,5,"""` mr .",4,0.46
5,0.2,6,he did consent .,5,0.48
6,0.2,7,he did consent .,5,0.48
7,0.2,8,he did consent .,5,0.48
8,0.2,9,he did consent .,5,0.48
9,0.2,10,he did consent .,5,0.48


,n_sentences,avg_perplexity,avg_tokens
temperature,,,
0.2,10,0.47,4.5
1.0,10,0.42,19.5
5.0,10,0.44,5.5


---
# end